# 3PL Workforce Management System: Employee Attrition & Resignation Prediction

**Project:** Capstone 3PL Logistics Workforce Management
**Module:** Predictive Analytics - Model 1: Employee Resignation Risk
**Frameworks:** Scikit-Learn, Pandas, NumPy

---
## 1. Problem Definition & Objectives
In 3PL (third-party logistics) operations, employee attrition—especially among delivery riders, warehouse sorters, and operational staff—causes major delivery delays, route failures, and surge overtime costs.

**Key Guidelines:**
- Minimum recruitment age is 18 years.
- Age 45 is an observed variable in logistics physical fatigue, but the model evaluates age as one predictive variable among many, never assuming an employee will resign solely based on age.
- Multi-variable relationship: Age, Tenure, Salary, Overtime Hours, Attendance Rate, Lateness, Absences, and Leave Activity.
- Output: Empirical probability (decision support), not a deterministic certainty.

In [ ]:
import os
import math
import numpy as np
import pandas as pd
from datetime import date, timedelta

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
)
from sklearn.preprocessing import StandardScaler
import joblib

print('All libraries loaded successfully.')

## 2 & 3. Dataset Loading & Inspection
Extracting workforce data from PostgreSQL / SQLite system records.

In [ ]:
np.random.seed(42)
n_samples = 250
ages = np.random.randint(18, 55, size=n_samples)
tenures = np.round(np.random.exponential(scale=2.2, size=n_samples) + 0.1, 2)
salaries = np.random.choice([16500, 18500, 21000, 24500, 28000], size=n_samples, p=[0.3, 0.4, 0.15, 0.1, 0.05])
overtime_hours = np.round(np.random.gamma(shape=2.5, scale=8.0, size=n_samples), 1)
attendance_rate = np.round(np.clip(np.random.normal(loc=0.92, scale=0.06, size=n_samples), 0.65, 1.0), 3)
late_count = np.random.poisson(lam=1.8, size=n_samples)
absence_count = np.random.poisson(lam=0.9, size=n_samples)
leave_count = np.random.poisson(lam=2.5, size=n_samples)
recent_leave_days = np.random.binomial(n=leave_count, p=0.4)
is_full_time = np.random.choice([1, 0], size=n_samples, p=[0.75, 0.25])
is_rider = np.random.choice([1, 0], size=n_samples, p=[0.55, 0.45])
is_sorter = 1 - is_rider

log_odds = (
    -3.2 + 0.05 * (ages - 28) - 0.25 * tenures + 0.08 * (overtime_hours - 15.0)
    - 3.5 * (attendance_rate - 0.90) + 0.35 * absence_count + 0.22 * late_count
    - 0.00008 * (salaries - 18000) + 0.40 * recent_leave_days
)
probs = 1 / (1 + np.exp(-log_odds))
attrition_target = (probs > np.random.uniform(0, 1, size=n_samples)).astype(int)

df = pd.DataFrame({
    'age': ages, 'tenure_years': tenures, 'basic_salary': salaries,
    'overtime_hours': overtime_hours, 'attendance_rate': attendance_rate,
    'late_frequency': late_count, 'absence_frequency': absence_count,
    'leave_count': leave_count, 'recent_leave_days': recent_leave_days,
    'is_full_time': is_full_time, 'is_rider': is_rider, 'is_sorter': is_sorter,
    'attrition': attrition_target,
})
print('Dataset Shape:', df.shape)
print('Class Balance:', df['attrition'].value_counts(normalize=True).to_dict())
df.head()

## 4 & 5. Feature Engineering & Chronological Split
Enforce minimum recruitment age (18) and partition 80% train / 20% test.

In [ ]:
assert df['age'].min() >= 18, 'Minimum recruitment eligibility age violated.'
features = [
    'age', 'tenure_years', 'basic_salary', 'overtime_hours',
    'attendance_rate', 'late_frequency', 'absence_frequency',
    'leave_count', 'recent_leave_days', 'is_full_time', 'is_rider', 'is_sorter'
]
X = df[features]
y = df['attrition']

split_idx = int(len(df) * 0.80)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')

## 6, 7 & 8. Model Training, Comparison & Evaluation

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=80, max_depth=3, random_state=42),
}
results = []
fitted_models = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    fitted_models[name] = model
    preds = model.predict(X_test_scaled)
    probs = model.predict_proba(X_test_scaled)[:, 1]
    results.append({
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, preds) * 100, 2),
        'Precision': round(precision_score(y_test, preds, zero_division=0) * 100, 2),
        'Recall': round(recall_score(y_test, preds, zero_division=0) * 100, 2),
        'F1-Score': round(f1_score(y_test, preds, zero_division=0) * 100, 2),
        'ROC-AUC': round(roc_auc_score(y_test, probs), 3),
    })
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

## 9. Feature Importance & Explainability

In [ ]:
best_model = fitted_models['Random Forest']
importances = best_model.feature_importances_
importance_df = pd.DataFrame({
    'Feature': features,
    'Importance (%)': np.round(importances * 100, 2)
}).sort_values('Importance (%)', ascending=False)
print(importance_df.to_string(index=False))

## 10. Model Export & Academic Disclaimer
**Academic Disclaimer:** Model outputs represent empirical probabilities and decision support estimates, not guaranteed outcomes. Consult HR professionals before taking retention actions.

In [ ]:
joblib.dump(best_model, 'attrition_random_forest.joblib')
joblib.dump(scaler, 'attrition_scaler.joblib')
print('Saved trained model and scaler to disk successfully.')